# Pose predictors

## File Handling
To run predictions a `RobotEnvironment` object and a `HeadsetData` object is needed, those can be loaded from folders or created.

### Creation of RobotEnvironment and HeadsetData
Those 2 datatypes can be created from an GatheredRobotData object and a .vrs file respectively.

In [ ]:
from headset_data import *
from robot_environment import *

In [ ]:
robot_data_folder_location = "/home/wmarx/AR-Headset-Localization-in-Robot-Scanned-Workspaces-A-Benchmark-Pipeline/datasets/r7_small_aruco"
vrs_file_location = "/home/wmarx/AR-Headset-Localization-in-Robot-Scanned-Workspaces-A-Benchmark-Pipeline/datasets/r7_small_aruco_5fps.vrs"


robot_data_from_disk = GatheredRobotData.from_folder(robot_data_folder_location)
robot_env_from_robot_data = RobotEnvironment.from_gathered_robot_data(
        robot_data = robot_data,
        number_of_sampled_datapoints=10,
        only_sample_robot_datapoints_w_marker_estimates = True,
        markers_use_advanced_removal=True,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig()
        est3d_xyz_icp_config=ICPAlignmentConfig()
)

headset_data_from_vrs = HeadsetData.from_vrs_file(vrs_file_location) 

### Adding labels to HeadsetData
To add labels to the HeadsetData for accuracy evaluation it has to be joined with an GatheredRobotData object with some supported marker (e.g. aruco/charuco). 
It can then be saved and does not need rebounding for labels (rebounding is still possible), so GatheredRobotData becomes obsolete

In [ ]:
# Labeling by using a robot_env_from_robot_data
labeled_headset_data = create_robot_bound_headset_data(
        headset_data = headset_data_from_vrs,
        robot_data = robot_data_from_disk
    )

### Saving and loading RobotEnvironments and HeadsetData

In [ ]:
processed_datasets_location = "/home/wmarx/AR-Headset-Localization-in-Robot-Scanned-Workspaces-A-Benchmark-Pipeline/processed_datasets"

labeled_headset_data.save(processed_datasets_location, name = "quickstart_headset_data")
robot_env_from_robot_data.save(processed_datasets_location, name = "quickstart_robot_data")

robot_env_from_disk = RobotEnvironment.from_folder(f"{processed_datasets_location}/quickstart_robot_data")
headset_data_from_disk = HeadsetData.from_folder(f"{headset_data_path}/quickstart_headset_data")

### Alternative: Creating from TU-München Dataset

In [ ]:
from load_from_tum import *

tum_rgbd_dataset_location = /home/wmarx/AR-Headset-Localization-in-Robot-Scanned-Workspaces-A-Benchmark-Pipeline/datasets/rgbd_dataset_freiburg2_desk

tum_robot_env, tum_headset_data = robot_environment_and_headset_data_from_tum(
        folder=tum_rgbd_dataset_location,
        rgb_camera_name="freiburg2",
        time_tolerance= 0.01,
        n_robot_images= 20,
        xyz_image_generation_config=XYZImageGenerationConfig(),
        xyz_image_alginment_config=ICPAlignmentConfig(),
)

## Testing Predictors

In [ ]:
from predictor_grader import *
from pose_pred_points import *
from pose_pred_points_lines import *
from pose_pred_points_ellipsoids import *

### Creating Predictors

In [ ]:
chosen_headset_data = headset_data_from_disk
chosen_robot_env = robot_env_from_disk


# Simple Point only predictor
point_predictor = NoExtrasPredictor(
        cam2_intrinsic_mtx=chosen_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=chosen_robot_env.robot_bgr_images,
        cam1_xyz_images=chosen_robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig()
)

### Grading the Performance of an Initialised Predictor:

In [ ]:
init_predictor_grade = PredictionOnDataset(
    predictor = point_predictor,
    headset_data = chosen_headset_data,
)